# 🚗 TakarMobil: Data Preprocessing & Pipeline Preparation
*TakarMobil: Sistem Estimasi Rentang Harga & Analisis Faktor Penentu Harga Mobil Bekas*

---

### 🎯 Tujuan Tahap Preprocessing (Phase 3):
1. **Data Cleaning & Parsing:** Menghapus data tanpa harga, menyaring harga wajar (Rp 25 Jt–10 M), menyaring 5 merek utama, parsing tahun (1995–2025), serta perbaikan parsing kilometer (menangani format `'K'` ribuan dari Mobil123).
2. **Pemetaan Wilayah Akurat & Deduplikasi:** Memetakan seluruh kawasan penyangga Jabodetabek termasuk format diskon Mobil123 (misal: *"Pondok Indah, 3% off"*, *"Bintaro, 5% off"*) serta menghapus duplikat `URL` dan iklan repost identik `(Judul, Merek, Tahun, Harga, Lokasi)`.
3. **Feature Engineering & Kunci Grup:** Membentuk `Usia_Mobil`, `Merek_Model`, target $\log(1 + \text{Harga})$, dan kunci grup iklan kembar `dup_group` (`Merek_Clean + Model_Clean + Tahun_Num + Harga_Num`).
4. **Skema Pembagian Data (`StratifiedGroupKFold` 80:20):** Membagi data berdasarkan grup iklan kembar (`groups=dup_group`) agar seluruh iklan dengan spesifikasi & harga identik masuk seluruhnya ke Train atau Test (**0 twin-ad leakage**), sekaligus menjaga distribusi tipe mobil (`Model_Clean`).
5. **Imputasi Missing Values Pasca-Split:** Menghitung median bertingkat `Kilometer` (`Model + Tahun` $\rightarrow$ `Merek + Tahun` $\rightarrow$ Global) **hanya dari data training**, lalu menerapkannya ke data training dan data testing (**0 leakage statistik test**).
6. **Pipeline Preprocessing:** `ColumnTransformer` (`StandardScaler` + `OneHotEncoder` termasuk `Model_Clean`) di-fit khusus pada `X_train`.

## 1. Import Library & Setup Lingkungan
Memuat pustaka standar untuk manipulasi data, scikit-learn untuk *preprocessing pipeline* dan *data splitting*, serta joblib untuk penyimpanan artifak preprocessor.


In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
print("Pustaka preprocessing berhasil dimuat!")

Pustaka preprocessing berhasil dimuat!


## 2. Pemuatan Dataset Mentah (Raw Data)
Memuat dataset mentah `Data Scraping.xlsx` menggunakan deteksi path dinamis yang fleksibel.


In [2]:
candidate_paths = [
    "data/raw/Data Scraping.xlsx",
    "../data/raw/Data Scraping.xlsx",
    "Data Scraping.xlsx"
]

file_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if file_path is None:
    raise FileNotFoundError("File Data Scraping.xlsx tidak ditemukan!")

print(f"Memuat dataset mentah dari: {file_path}")
df_raw = pd.read_excel(file_path)
print(f"Jumlah baris awal: {df_raw.shape[0]}, Jumlah kolom: {df_raw.shape[1]}")
df_raw.head(3)


Memuat dataset mentah dari: ../data/raw/Data Scraping.xlsx


Jumlah baris awal: 17670, Jumlah kolom: 16


,Sumber,URL,Judul,Merek,Model,Varian,Tahun,Kilometer,Transmisi,Warna,Kapasitas Mesin,Bahan Bakar,Tanggal Upload,Lokasi,Tipe Bodi,Harga
0,OLX,https://www.olx.co.id/item/939502279,MERCEDES BENZ A45 AMG 2014 2.0 TURBO AWD PANOR...,Mercedes-Benz,A45 AMG,2.0 Bensin-AT,2014,NaN,Automatic,Putih,NaN,Bensin,07-10-2025,Jakarta Timur,NaN,499.000.000
1,OLX,https://www.olx.co.id/item/939871468,Mercedes Benz Sprinter A415 CDI 2022,Mercedes-Benz,Sprinter,NaN,2021,NaN,Manual,Hitam,NaN,Diesel,17-10-2025,Jakarta Utara,NaN,1.850.000.000
2,OLX,https://www.olx.co.id/item/939450820,[ GARANSI 5TH ] Mercedes Benz Mercy GLC200 GLC...,Mercedes-Benz,GLC200,2.0 AMG Line Bensin-AT,2022,NaN,Automatic,Putih,NaN,Bensin,06-10-2025,Jakarta Barat,NaN,720.000.000


## 3. Data Cleaning & Standarisasi Fitur
Pada tahap ini kita menyelesaikan berbagai anomali teks dan artefak web scraping yang telah ditemukan pada tahap EDA:
1. **Drop Kolom Tak Relevan:** Kolom `URL`, `Judul`, `Tanggal Upload`, dan `Tipe Bodi` (karena >47% kosong) dihapus dari fitur pemodelan.
2. **Parsing Harga:** Mengonversi string format Rupiah (bertitik) menjadi float murni, lalu menghapus data tanpa harga.
3. **Penyaringan Outlier Ekstrem:** Membatasi harga mobil bekas pada rentang rasional (Rp 25 Juta s/d Rp 10 Miliar).
4. **Standarisasi Merek:** Memperbaiki kesalahan scraping di mana tipe seri Mercedes (seperti `C-Class`, `E-Class`, `GLC-Class`, `GLA-Class`) terdata sebagai merek terpisah.
5. **Penyatuan Bahan Bakar:** Menyatukan label `Bensin` dan `Petrol` (sinonim dari dua platform scraping berbeda) menjadi satu kategori: `Bensin`.
6. **Parsing Kilometer & Tahun:** Mengambil nilai tengah dari rentang teks kilometer dan memvalidasi rentang tahun kendaraan.


In [3]:
df = df_raw.copy()

# A. Drop Baris tanpa Harga & Parsing Harga
def clean_price(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    digits = ''.join(c for c in str(val).replace('.', '').strip() if c.isdigit())
    return float(digits) if digits else np.nan

df['Harga_Num'] = df['Harga'].apply(clean_price)
df = df.dropna(subset=['Harga_Num'])

# Filter rentang harga rasional (25 Juta - 10 Miliar)
df = df[(df['Harga_Num'] >= 25_000_000) & (df['Harga_Num'] <= 10_000_000_000)]
print(f"Jumlah data setelah pembersihan & filter harga: {len(df)} baris")


Jumlah data setelah pembersihan & filter harga: 17363 baris


In [4]:
# B. Standarisasi Kolom Merek
# Kamus pemetaan seri mobil Mercedes-Benz yang terpisah saat scraping
mercedes_series = [
    'C-Class', 'E-Class', 'GLC-Class', 'GLA-Class', 'CLA-Class', 'GLE-Class',
    'A-Class', 'S-Class', 'GLS-Class', 'V-Class', 'CLS-Class', 'SLK-Class',
    'GLB-Class', 'B-Class', 'ML-Class', 'G-Class', 'GL-Class', 'CLS400',
    'CLA200', 'SL-Class', 'Mercedes Benz'
]

# Fungsi standarisasi merek
def standardize_brand(m):
    if pd.isna(m):
        return 'Lainnya'
    m_str = str(m).strip()
    if m_str in mercedes_series or 'mercedes' in m_str.lower() or 'benz' in m_str.lower():
        return 'Mercedes-Benz'
    if 'toyota' in m_str.lower():
        return 'Toyota'
    if 'daihatsu' in m_str.lower():
        return 'Daihatsu'
    if 'suzuki' in m_str.lower():
        return 'Suzuki'
    if 'honda' in m_str.lower():
        return 'Honda'
    return m_str

df['Merek_Clean'] = df['Merek'].apply(standardize_brand)

# Filter hanya 5 Merek Utama Proyek
TARGET_BRANDS = ['Toyota', 'Daihatsu', 'Suzuki', 'Honda', 'Mercedes-Benz']
df = df[df['Merek_Clean'].isin(TARGET_BRANDS)]

print("Distribusi 5 Merek Utama setelah standarisasi:")
print(df['Merek_Clean'].value_counts())


Distribusi 5 Merek Utama setelah standarisasi:
Merek_Clean
Toyota           3748
Daihatsu         3444
Mercedes-Benz    3347
Suzuki           3222
Honda            3079
Name: count, dtype: int64


In [5]:
# C. Standarisasi Transmisi
def clean_transmission(val):
    if pd.isna(val): return 'Automatic'
    v = str(val).lower()
    if 'auto' in v or 'at' in v: return 'Automatic'
    if 'man' in v or 'mt' in v: return 'Manual'
    return 'Automatic'
df['Transmisi_Clean'] = df['Transmisi'].apply(clean_transmission)

# D. Standarisasi Bahan Bakar
def clean_fuel(val):
    if pd.isna(val): return 'Bensin'
    v = str(val).lower()
    if 'bensin' in v or 'petrol' in v or 'gasoline' in v: return 'Bensin'
    if 'diesel' in v or 'solar' in v: return 'Diesel'
    if 'hybrid' in v or 'listrik' in v or 'electric' in v: return 'Hybrid'
    return 'Bensin'
df['Bahan_Bakar_Clean'] = df['Bahan Bakar'].apply(clean_fuel)

# E. Parsing Kilometer (menangani format '10 - 15K km' dari Mobil123 & '50.000-55.000 km' dari OLX)
def clean_km(val):
    if pd.isna(val): return np.nan
    s = str(val).lower().replace('km', '').strip()
    is_k = 'k' in s
    s = s.replace('k', '').strip()
    nums = []
    for p in s.split('-'):
        d = ''.join(c for c in p.replace('.', '').replace(',', '').strip() if c.isdigit())
        if d:
            v = float(d)
            if is_k or v <= 500:
                v *= 1000.0
            nums.append(v)
    if not nums: return np.nan
    res = float(np.mean(nums))
    return res if 0 <= res <= 400_000 else np.nan
df['KM_Num'] = df['Kilometer'].apply(clean_km)

# F. Parsing Tahun
def clean_year(val):
    if pd.isna(val): return np.nan
    d = ''.join(c for c in str(val) if c.isdigit())
    if d and len(d) == 4:
        y = int(d)
        if 1995 <= y <= 2025: return y
    return np.nan
df['Tahun_Num'] = df['Tahun'].apply(clean_year)
df = df.dropna(subset=['Tahun_Num'])
df['Tahun_Num'] = df['Tahun_Num'].astype(int)

# G. Standarisasi Model Mobil
df['Model_Clean'] = df['Model'].fillna('Lainnya').astype(str).str.strip().str.title()

# H. Simplifikasi Wilayah Lokasi (menangani nama kecamatan/kawasan berdiskon '% off' dari Mobil123)
TANGERANG_KEYWORDS = [
    'tangerang', 'alam sutera', 'pamulang', 'banjar wijaya', 'batu ceper',
    'bsd', 'cikupa', 'ciledug', 'cipondoh', 'ciputat', 'citra raya',
    'gading serpong', 'jatake', 'jatiuwung', 'jurangmangu', 'karawaci',
    'pondok cabe', 'serpong'
]
BEKASI_KEYWORDS = [
    'bekasi', 'bintara', 'cibitung', 'harapan indah', 'jatiwaringin',
    'jatiwarna', 'kartini', 'medan satria', 'pondok gede'
]
DEPOK_KEYWORDS = ['depok', 'cimanggis', 'kelapa dua']
BOGOR_KEYWORDS = ['bogor', 'cibinong', 'cibubur', 'cileungsi']
JAKARTA_KEYWORDS = [
    'jakarta', 'ancol', 'batu tulis', 'bintaro', 'buaran', 'cakung',
    'cempaka putih', 'cengkareng', 'cijantung', 'cilandak', 'cilincing',
    'cipinang', 'ciracas', 'daan mogot', 'dewi sartika', 'duren sawit',
    'duri kosambi', 'fatmawati', 'grogol', 'gunung sahari', 'jalan panjang',
    'kalideres', 'kebayoran', 'kebon jeruk', 'kelapa gading', 'kemang',
    'kemayoran', 'kramat jati', 'kuningan', 'lebak bulus', 'lubang buaya',
    'mangga besar', 'mangga dua', 'pademangan', 'pancoran',
    'pantai indah kapuk', 'pasar minggu', 'pasar rebo', 'permata hijau',
    'pluit', 'pondok indah', 'pulo gebang', 'puri indah', 'sawah besar',
    'setiabudi', 'sunter', 'tamansari', 'taman sari', 'tambora'
]

def simplify_location(loc):
    if pd.isna(loc): return 'Jakarta'
    l = str(loc).lower()
    if any(k in l for k in TANGERANG_KEYWORDS): return 'Tangerang'
    if any(k in l for k in BEKASI_KEYWORDS): return 'Bekasi'
    if any(k in l for k in DEPOK_KEYWORDS): return 'Depok'
    if any(k in l for k in BOGOR_KEYWORDS): return 'Bogor'
    if any(k in l for k in JAKARTA_KEYWORDS): return 'Jakarta'
    return 'Luar Jabodetabek'

df['Wilayah'] = df['Lokasi'].apply(simplify_location)

# I. Deduplikasi Data Ganda (URL & Iklan Repost Identik)
before_dedup = len(df)
df = df.drop_duplicates(subset=['URL']).reset_index(drop=True)
after_url = len(df)
df = df.drop_duplicates(subset=['Judul', 'Merek_Clean', 'Tahun_Num', 'Harga_Num', 'Lokasi']).reset_index(drop=True)

print(f"Sebelum deduplikasi : {before_dedup} baris")
print(f"Setelah hapus URL ganda : {after_url} baris (-{before_dedup - after_url})")
print(f"Setelah hapus iklan identik (Judul+Merek+Tahun+Harga+Lokasi): {len(df)} baris (-{after_url - len(df)})")
print("Distribusi Wilayah hasil pemetaan:")
print(df['Wilayah'].value_counts())
df[['Merek_Clean', 'Model_Clean', 'Tahun_Num', 'KM_Num', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah', 'Harga_Num']].head()

Sebelum deduplikasi : 16799 baris
Setelah hapus URL ganda : 14754 baris (-2045)
Setelah hapus iklan identik (Judul+Merek+Tahun+Harga+Lokasi): 13359 baris (-1395)
Distribusi Wilayah hasil pemetaan:
Wilayah
Jakarta             8857
Tangerang           1536
Bekasi               999
Bogor                811
Luar Jabodetabek     682
Depok                474
Name: count, dtype: int64


,Merek_Clean,Model_Clean,Tahun_Num,KM_Num,Transmisi_Clean,Bahan_Bakar_Clean,Wilayah,Harga_Num
0,Mercedes-Benz,A45 Amg,2014,NaN,Automatic,Bensin,Jakarta,4.990000e+08
1,Mercedes-Benz,Sprinter,2021,NaN,Manual,Diesel,Jakarta,1.850000e+09
2,Mercedes-Benz,Glc200,2022,NaN,Automatic,Bensin,Jakarta,7.200000e+08
3,Mercedes-Benz,Gle400,2016,NaN,Automatic,Bensin,Jakarta,5.980000e+08
4,Mercedes-Benz,Lainnya,2010,NaN,Automatic,Bensin,Luar Jabodetabek,1.590000e+08


## 4. Feature Engineering & Kunci Grup Anti-Leakage
Membentuk fitur turunan dan kunci grup sebelum pembagian train/test:
1. **`Usia_Mobil`:** Selisih tahun acuan (2025) dengan tahun perakitan mobil ($2025 - \text{Tahun}$).
2. **`Log_Harga`:** Transformasi logaritma natural $\log(1 + \text{Harga})$ sebagai target pelatihan model regresi.
3. **`Merek_Model`:** Gabungan merek dan model/tipe spesifik (`Model_Clean`) untuk menjaga keterwakilan tipe kendaraan.
4. **`dup_group`:** Kunci grup iklan kembar (`Merek_Clean + Model_Clean + Tahun_Num + Harga_Num`) agar iklan kembar lintas platform atau judul sedikit berbeda tidak pernah terpisah antara data training dan testing.

In [6]:
# 1. Usia Mobil
df['Usia_Mobil'] = 2025 - df['Tahun_Num']

# 2. Target Log-Transformed
df['Log_Harga'] = np.log1p(df['Harga_Num'])

# 3. Kunci Stratifikasi Merek + Tipe
df['Merek_Model'] = df['Merek_Clean'] + "_" + df['Model_Clean']

# 4. Kunci Grup Iklan Kembar (Merek + Model + Tahun + Harga)
df['dup_group'] = (
    df['Merek_Clean'] + "_" +
    df['Model_Clean'] + "_" +
    df['Tahun_Num'].astype(str) + "_" +
    df['Harga_Num'].astype('int64').astype(str)
)

print(f"Total iklan unik: {len(df)} | Jumlah grup unik (Merek+Model+Tahun+Harga): {df['dup_group'].nunique()}")
df[['Merek_Clean', 'Model_Clean', 'Tahun_Num', 'Usia_Mobil', 'KM_Num', 'Wilayah', 'Harga_Num', 'Log_Harga', 'dup_group']].head()

Total iklan unik: 13359 | Jumlah grup unik (Merek+Model+Tahun+Harga): 7635


,Merek_Clean,Model_Clean,Tahun_Num,Usia_Mobil,KM_Num,Wilayah,Harga_Num,Log_Harga,dup_group
0,Mercedes-Benz,A45 Amg,2014,11,NaN,Jakarta,4.990000e+08,20.028117,Mercedes-Benz_A45 Amg_2014_499000000
1,Mercedes-Benz,Sprinter,2021,4,NaN,Jakarta,1.850000e+09,21.338451,Mercedes-Benz_Sprinter_2021_1850000000
2,Mercedes-Benz,Glc200,2022,3,NaN,Jakarta,7.200000e+08,20.394762,Mercedes-Benz_Glc200_2022_720000000
3,Mercedes-Benz,Gle400,2016,9,NaN,Jakarta,5.980000e+08,20.209101,Mercedes-Benz_Gle400_2016_598000000
4,Mercedes-Benz,Lainnya,2010,15,NaN,Luar Jabodetabek,1.590000e+08,18.884415,Mercedes-Benz_Lainnya_2010_159000000


## 5. Pembagian Data: `StratifiedGroupKFold` (80% Train : 20% Test, Anti-Leakage)
Sesuai prinsip *no test leakage*, pembagian dataset dilakukan **terlebih dahulu sebelum imputasi missing values**:
- Menggunakan **`StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`** dengan **`groups = dup_group`**.
- Stratifikasi target menggunakan `Merek_Model` (untuk tipe dengan $\ge 5$ grup unik, sedangkan varian sangat langka digabung ke `'RARE_MODEL'`).
- Dengan `groups = dup_group`, seluruh iklan dengan kombinasi `(Merek, Model, Tahun, Harga)` yang identik diisolasi sepenuhnya ke Train **atau** ke Test (**0 grup bocor**), bukan sekadar membagi baris secara acak.

In [7]:
grp_counts = df.groupby('Merek_Model')['dup_group'].nunique()
strat_key = df['Merek_Model'].where(df['Merek_Model'].map(grp_counts) >= 5, 'RARE_MODEL')

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, strat_key, groups=df['dup_group']))

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

overlap_groups = set(train_df['dup_group']) & set(test_df['dup_group'])
assert len(overlap_groups) == 0, f"Leakage terdeteksi! Overlap grup: {len(overlap_groups)}"

print(f"Total Data Bersih : {len(df)} baris")
print(f"Data Training     : {len(train_df)} baris ({len(train_df)/len(df)*100:.2f}%)")
print(f"Data Testing      : {len(test_df)} baris ({len(test_df)/len(df)*100:.2f}%)")
print(f"Overlap dup_group antara Train & Test: {len(overlap_groups)} (0 Twin-Ad Leakage Verified)")
print(f"Jumlah tipe mobil (Model_Clean) unik di Train: {train_df['Model_Clean'].nunique()} | di Test: {test_df['Model_Clean'].nunique()}")

Total Data Bersih : 13359 baris
Data Training     : 10690 baris (80.02%)
Data Testing      : 2669 baris (19.98%)
Overlap dup_group antara Train & Test: 0 (0 Twin-Ad Leakage Verified)
Jumlah tipe mobil (Model_Clean) unik di Train: 211 | di Test: 158


## 6. Penanganan Missing Values (Imputasi KM Pasca-Split Khusus Data Training)
Kolom `KM_Num` memiliki nilai kosong sekitar 20%. Untuk memastikan **nol kebocoran statistik data test**, tabel median kilometer dihitung **secara eksklusif hanya dari `train_df`** dengan hierarki bertingkat:
1. Tingkat 1: Nilai median `KM_Num` pada **`train_df`** berdasarkan kombinasi `Model_Clean` + `Tahun_Num`.
2. Tingkat 2 (*Fallback*): Jika kombinasi model & tahun tersebut tidak ada di data training, gunakan median `KM_Num` pada **`train_df`** berdasarkan `Merek_Clean` + `Tahun_Num`.
3. Tingkat 3 (*Global Fallback*): Nilai median global `KM_Num` dari **`train_df`**.

In [8]:
print(f"Missing KM sebelum imputasi -> Train: {train_df['KM_Num'].isnull().sum()} | Test: {test_df['KM_Num'].isnull().sum()}")

# Hitung lookup tabel median HANYA dari data training!
km_med_model_year = train_df.groupby(['Model_Clean', 'Tahun_Num'])['KM_Num'].median().to_dict()
km_med_brand_year = train_df.groupby(['Merek_Clean', 'Tahun_Num'])['KM_Num'].median().to_dict()
km_med_global = float(train_df['KM_Num'].median())

def apply_train_km_imputation(part_df):
    my_fill = pd.Series(
        [km_med_model_year.get((m, t), np.nan) for m, t in zip(part_df['Model_Clean'], part_df['Tahun_Num'])],
        index=part_df.index,
        dtype='float64'
    )
    by_fill = pd.Series(
        [km_med_brand_year.get((b, t), np.nan) for b, t in zip(part_df['Merek_Clean'], part_df['Tahun_Num'])],
        index=part_df.index,
        dtype='float64'
    )
    return part_df['KM_Num'].fillna(my_fill).fillna(by_fill).fillna(km_med_global)

train_df['KM_Clean'] = apply_train_km_imputation(train_df)
test_df['KM_Clean'] = apply_train_km_imputation(test_df)
df['KM_Clean'] = apply_train_km_imputation(df)

assert train_df['KM_Clean'].isnull().sum() == 0
assert test_df['KM_Clean'].isnull().sum() == 0
print(f"Missing KM setelah imputasi  -> Train: {train_df['KM_Clean'].isnull().sum()} | Test: {test_df['KM_Clean'].isnull().sum()}")
print(f"Global median KM (khusus dari train_df): {km_med_global:,.0f} KM (0 test statistics used)")

Missing KM sebelum imputasi -> Train: 2351 | Test: 582
Missing KM setelah imputasi  -> Train: 0 | Test: 0
Global median KM (khusus dari train_df): 52,500 KM (0 test statistics used)


## 7. Membangun Pipeline Preprocessing (Scikit-Learn ColumnTransformer)
Mempersiapkan transformer yang terintegrasi untuk standarisasi skala numerik dan *One-Hot Encoding* variabel kategorikal. Transformer ini di-fit **hanya** pada data training untuk menjamin kaidah machine learning yang benar.


In [9]:
num_features = ['Tahun_Num', 'Usia_Mobil', 'KM_Clean']
cat_features = ['Merek_Clean', 'Model_Clean', 'Transmisi_Clean', 'Bahan_Bakar_Clean', 'Wilayah']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ],
    remainder='drop'
)

# Fit preprocessor pada data training
X_train = train_df[num_features + cat_features]
y_train = train_df['Log_Harga']

X_test = test_df[num_features + cat_features]
y_test = test_df['Log_Harga']

preprocessor.fit(X_train)
print("ColumnTransformer berhasil di-fit pada data training.")

# Cek dimensi matriks fitur hasil transformasi
X_train_transformed = preprocessor.transform(X_train)
print(f"Bentuk matriks fitur setelah transformasi: {X_train_transformed.shape[0]} baris x {X_train_transformed.shape[1]} kolom")

ColumnTransformer berhasil di-fit pada data training.
Bentuk matriks fitur setelah transformasi: 10690 baris x 230 kolom


## 8. Ekspor Dataset Olahan & Artefak Preprocessing
Menyimpan dataset yang telah bersih dan siap pakai ke direktori `data/processed/`, serta menyimpan preprocessor ter-fit ke folder `models/`.


In [10]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('models', exist_ok=True)

proc_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
models_dir = '../models' if os.path.exists('../models') else 'models'

clean_path = os.path.join(proc_dir, 'clean_cars_data.csv')
df.to_csv(clean_path, index=False)
print(f"Data bersih disimpan ke: {clean_path}")

train_path = os.path.join(proc_dir, 'train_data.csv')
test_path = os.path.join(proc_dir, 'test_data.csv')
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)
print(f"Data train disimpan ke : {train_path}")
print(f"Data test disimpan ke  : {test_path}")

preproc_path = os.path.join(models_dir, 'preprocessor.joblib')
joblib.dump(preprocessor, preproc_path)
print(f"Preprocessor disimpan ke: {preproc_path}")

models_by_brand = {}
for brand, grp in df.groupby('Merek_Clean'):
    vc_b = grp['Model_Clean'].value_counts()
    models_by_brand[brand] = [{'name': str(m), 'count': int(c)} for m, c in vc_b.items()]

wilayah_counts = {str(k): int(v) for k, v in df['Wilayah'].value_counts().items()}

metadata = {
    'num_features': num_features,
    'cat_features': cat_features,
    'target_column': 'Harga_Num',
    'log_target_column': 'Log_Harga',
    'brands': TARGET_BRANDS,
    'models_by_brand': models_by_brand,
    'wilayah_list': sorted(df['Wilayah'].unique().tolist()),
    'wilayah_counts': wilayah_counts,
    'transmisi_list': sorted(df['Transmisi_Clean'].unique().tolist()),
    'bahan_bakar_list': sorted(df['Bahan_Bakar_Clean'].unique().tolist()),
    'total_clean_samples': int(len(df)),
    'train_samples': int(len(train_df)),
    'test_samples': int(len(test_df))
}

meta_path = os.path.join(models_dir, 'feature_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata fitur disimpan ke: {meta_path}")

Data bersih disimpan ke: ../data/processed\clean_cars_data.csv


Data train disimpan ke : ../data/processed\train_data.csv
Data test disimpan ke  : ../data/processed\test_data.csv
Preprocessor disimpan ke: ../models\preprocessor.joblib
Metadata fitur disimpan ke: ../models\feature_metadata.json


## 9. Ringkasan Hasil Preprocessing (Checklist Phase 3)

| Komponen Preprocessing | Status | Keterangan / Implementasi |
|---|---|---|
| **Data Cleaning** | ✅ Selesai | Filter 5 merek utama, unifikasi bensin/petrol, standarisasi seri Mercedes, perbaikan parsing KM `'K'`, serta pemetaan lengkap kawasan Jabodetabek termasuk label `'% off'`. |
| **Deduplikasi Data** | ✅ Selesai | Hapus duplikat `URL` & iklan repost identik (`Judul + Merek + Tahun + Harga + Lokasi`). |
| **Pembersihan Target** | ✅ Selesai | Drop NaN target, hapus outlier non-rasional (< Rp 25 Jt & > Rp 10 M). |
| **Feature Engineering** | ✅ Selesai | Pembentukan `Usia_Mobil`, `Wilayah`, `Merek_Model`, `Model_Clean`, `dup_group`, dan target `Log_Harga`. |
| **Validasi & Splitting** | ✅ Selesai | **`StratifiedGroupKFold` (80:20)** berbasis grup iklan kembar `dup_group` (*0 twin-ad leakage*). |
| **Imputasi Missing Values** | ✅ Selesai | Imputasi bertingkat `KM_Num` dilakukan **pasca-split hanya menggunakan statistik `train_df`** (*0 test leakage*). |
| **Pipeline Scikit-Learn** | ✅ Selesai | `ColumnTransformer` (`StandardScaler` + `OneHotEncoder` termasuk `Model_Clean`) di-fit khusus pada `X_train`. |